# TDA6323 Clustering Project - CUDA Experiment Notebook

This notebook runs the Part 2 clustering experiment for **K-Means**, **DBSCAN**, and **Agglomerative Hierarchical Clustering**.

It uses the separated Python algorithm files in `Part2/programs/` and runs on CUDA when a working CUDA backend is available.

## 1. Setup

Run this cell first. If CUDA is available through PyTorch, the notebook will use your NVIDIA GPU. If not, it will fall back to CPU.

In [ ]:
from pathlib import Path
import sys
import time
import csv

import numpy as np

# Find the project/program folder whether the notebook is opened from TDA or Part2.
cwd = Path.cwd()
if (cwd / "Part2" / "programs").exists():
    PROJECT_ROOT = cwd
    PROGRAM_DIR = cwd / "Part2" / "programs"
    RESULTS_DIR = cwd / "Part2" / "results"
elif (cwd / "programs").exists():
    PROJECT_ROOT = cwd.parent
    PROGRAM_DIR = cwd / "programs"
    RESULTS_DIR = cwd / "results"
else:
    raise FileNotFoundError("Cannot find Part2/programs. Open this notebook from the TDA or Part2 folder.")

sys.path.insert(0, str(PROGRAM_DIR))

from cuda_backend import get_backend
from kmeans_algorithm import generate_clustering_problem, kmeans
from dbscan_algorithm import dbscan
from hierarchical_algorithm import hierarchical_agglomerative

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda"  # Change to "cpu" if you want CPU only.
backend = get_backend(DEVICE)
print("Project root:", PROJECT_ROOT)
print("Program folder:", PROGRAM_DIR)
print("Results folder:", RESULTS_DIR)
print("Device:", backend.name)
print(backend.note)

## 2. Clustering Problem

The experiment generates unlabelled two-dimensional data with four natural clusters and scattered noise-like points. Each algorithm solves the same grouping problem using a different design approach.

In [ ]:
SIZES = (200, 400, 800, 1200, 1600)
REPEAT = 2

datasets = {size: generate_clustering_problem(size) for size in SIZES}

print("Input sizes:", SIZES)
print("Repeats per algorithm and size:", REPEAT)
print("Smallest dataset shape:", datasets[SIZES[0]].shape)
print("Largest dataset shape:", datasets[SIZES[-1]].shape)

## 3. Helper Functions

In [ ]:
def count_clusters_and_noise(labels):
    labels = np.asarray(labels)
    clusters = len(set(int(label) for label in labels if int(label) != -1))
    noise = int(np.sum(labels == -1))
    return clusters, noise


def time_one_algorithm(name, data, run_index=0):
    start = time.perf_counter()
    if name == "K-Means":
        labels = kmeans(data, k=4, seed=42 + run_index, device=DEVICE)
    elif name == "DBSCAN":
        labels = dbscan(data, eps=0.34, min_samples=5, device=DEVICE)
    elif name == "Hierarchical":
        labels = hierarchical_agglomerative(data, k=4, device=DEVICE)
    else:
        raise ValueError(name)
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    clusters, noise = count_clusters_and_noise(labels)
    return elapsed_ms, clusters, noise


def run_algorithm_section(name, explanation):
    print("=" * 78)
    print(name)
    print("=" * 78)
    print(explanation)
    rows = []
    for size in SIZES:
        data = datasets[size]
        measurements = [time_one_algorithm(name, data, run_index=i) for i in range(REPEAT)]
        measurements = sorted(measurements, key=lambda item: item[0])
        elapsed_ms, clusters, noise = measurements[len(measurements) // 2]
        rows.append({
            "algorithm": name,
            "input_size": size,
            "elapsed_ms": elapsed_ms,
            "clusters_found": clusters,
            "noise_points": noise,
        })
        print(f"n={size:4d} time={elapsed_ms:10.3f} ms clusters={clusters} noise={noise}")
    first, last = rows[0], rows[-1]
    ratio = last["elapsed_ms"] / first["elapsed_ms"] if first["elapsed_ms"] else 0
    print(f"Finding: size increased {last['input_size'] / first['input_size']:.1f}x; runtime increased about {ratio:.1f}x.")
    print()
    return rows

## 4. Run the Three Algorithms

In [ ]:
# CUDA warm-up to avoid counting first GPU kernel startup cost.
if backend.using_cuda:
    warmup = generate_clustering_problem(32)
    kmeans(warmup, k=4, device=DEVICE)
    dbscan(warmup, eps=0.34, min_samples=5, device=DEVICE)
    hierarchical_agglomerative(warmup, k=4, device=DEVICE)
    print("CUDA warm-up completed.")

In [ ]:
all_results = []

all_results += run_algorithm_section(
    "K-Means",
    "K-Means solves the problem by assigning each point to the nearest centroid, then updating centroids until stable."
)

all_results += run_algorithm_section(
    "DBSCAN",
    "DBSCAN solves the problem by finding dense neighbourhoods and marking sparse points as noise."
)

all_results += run_algorithm_section(
    "Hierarchical",
    "Hierarchical clustering solves the problem by merging the nearest clusters until four final clusters remain."
)

len(all_results)

## 5. Save Notebook Results

In [ ]:
csv_path = RESULTS_DIR / "notebook_execution_times.csv"

with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["algorithm", "input_size", "elapsed_ms", "clusters_found", "noise_points"])
    writer.writeheader()
    for row in all_results:
        writer.writerow({
            "algorithm": row["algorithm"],
            "input_size": row["input_size"],
            "elapsed_ms": f"{row['elapsed_ms']:.6f}",
            "clusters_found": row["clusters_found"],
            "noise_points": row["noise_points"],
        })

print("Saved:", csv_path)

## 6. Summary

- **K-Means** is normally fastest for compact rounded clusters, but it does not identify noise.
- **DBSCAN** can identify noise points, but its runtime grows because it checks neighbourhoods.
- **Hierarchical clustering** gives an interpretable merge process, but it becomes expensive as input size grows.
- With CUDA enabled, distance-heavy operations can be accelerated on the GPU.